In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from utils import plot_returns

## Asset Universe

The portfolio is built from four diversified asset classes, each represented by a liquid instrument:

| Ticker | Asset Class | Description |
|---|---|---|
| `ES=F` | Equities | S&P 500 E-mini Futures — tracks US large-cap equity performance |
| `ZN=F` | Fixed Income | 10-Year US Treasury Note Futures — proxy for long-duration government bonds |
| `GC=F` | Commodities | Gold Futures — inflation hedge and safe-haven asset |
| `DX-Y.NYB` | Currencies | ICE US Dollar Index — measures the dollar against a basket of major currencies |

These four assets have historically low or negative correlations with each other, making them well-suited for a **risk parity** strategy.

In [ ]:
# Download front-month futures data of S&P500, 10-year Treasuries, gold and US dollar
symbols = ["ES=F", "ZN=F", "GC=F", "DX-Y.NYB"]
data = yf.download(symbols, period="10y")
# Resample data so that we deal with monthly data instead of daily to reduce noise
data = data.resample("ME").last()
data.index = pd.to_datetime(data.index)
# Subset adjusted close prices, replace zeros (yfinance missing data sentinel) with NaN,
# forward-fill, and drop rows with unknown prices in the beginning of the dataset
prices = data["Close"].ffill().dropna()
prices.index = pd.to_datetime(prices.index)

In [ ]:
prices

In [ ]:
# Compute logarithmic returns
log_returns = np.log(prices).diff()
# Compute annualized volatility: std of monthly log returns scaled by sqrt(12)
number_months_year = 12
annual_vol = log_returns.std() * np.sqrt(number_months_year)
inverse_vol = 1 / annual_vol
risk_parity_weights = inverse_vol / inverse_vol.sum()
# np.exp(log_returns) gives gross returns (~1.01 for a 1% gain); subtract 1 to get net
# simple returns (~0.01). min_count=1 prevents NaN rows collapsing to 0 before np.log.
weighted_returns = (np.exp(log_returns) * risk_parity_weights).sum(
    axis=1, min_count=1
) - 1

In [ ]:
log_returns.head()

In [ ]:
annual_vol

In [ ]:
inverse_vol

In [ ]:
risk_parity_weights

In [ ]:
weighted_returns

In [ ]:
# Annualized returns and volatility
number_months_year = 12
annualized_return = weighted_returns.mean() * number_months_year
annualized_vol = weighted_returns.std() * np.sqrt(number_months_year)

In [ ]:
print(f"Annualized Return: {annualized_return * 100:.2f}%")
print(f"Annualized Volatility: {annualized_vol * 100:.2f}%")

In [ ]:
# ^IRX is the 3-month T-bill yield, already reported as an annualized % (e.g. 5.2 means 5.2%)
# Divide by 100 to convert to decimal, then take the mean over the same 10-year window
tbill = yf.download("^IRX", period="10y")["Close"].resample("ME").last().ffill()
risk_free_annualized_return = (tbill / 100).mean().item()
print(f"Risk-free annualized return: {risk_free_annualized_return * 100:.2f}%")

In [ ]:
tbill

In [ ]:
# Calculate the Sharpe Ratio
sharpe_ratio = (annualized_return - risk_free_annualized_return) / annualized_vol
# Calculate the Sortino Ratio
# Clip positive returns to 0, keeping the full sample size N in the denominator
downside_vol = np.sqrt((weighted_returns.clip(upper=0) ** 2).mean()) * np.sqrt(
    number_months_year
)
sortino_ratio = (annualized_return - risk_free_annualized_return) / downside_vol
# Calculate the Calmar Ratio
# weighted_returns are simple returns, so compound with cumprod not cumsum
cum_returns = (1 + weighted_returns).cumprod()
# Drawdown = drop from the running peak of the cumulative wealth curve
drawdowns = (cum_returns - cum_returns.cummax()) / cum_returns.cummax()
max_drawdown = drawdowns.min()  # most negative value = worst drawdown
calmar_ratio = annualized_return / np.abs(max_drawdown)

In [ ]:
print()
print(f"annualized_return: {np.round(annualized_return * 100, 1)}")
print(f"annualized_volatility: {np.round(annualized_vol * 100, 1)}")
print(f"downside_volatility: {np.round(downside_vol * 100, 1)}%")
print(f"max_drawdown: {np.round(max_drawdown * 100, 1)}%")
print()
print(f"sharpe_ratio: {np.round(sharpe_ratio, 2)}")
print(f"sortino_ratio: {np.round(sortino_ratio, 2)}")
print(f"calmar_ratio: {np.round(calmar_ratio, 2)}")
print()

In [ ]:
plot_returns(weighted_returns)

In [ ]:
cum_returns.tail()